# 12.11 - Text Splitters

**Phase:** 12 - LangChain

**Status:** VERIFIED

---

## 1. What Are We Solving?

LLMs have context limits. Large documents must be split into chunks.

## 2. Why Does This Matter?

Bad splitting = lost context = bad retrieval = bad answers.

## 3. Prerequisites

- 12.10: Document loaders

## 4. Learning Objectives

- Split text into chunks
- Configure chunk_size and chunk_overlap
- Understand splitting strategies

## 5. Mental Model

Splitter = break text into chunks.
Recursive tries: paragraph, then sentence, then word.
Overlap ensures context between chunks.

In [1]:
from langchain_core.documents import Document
print("Document imported.")

Document imported.


## 6. Basic Splitting

In [2]:
sample = "LangChain is a framework for building LLM applications. It provides tools for prompt management, chaining, memory, and more.\n\nLangChain supports multiple LLM providers including OpenAI, Anthropic, and Groq. The framework is designed for production use.\n\nThe key abstractions are Chains, Agents, and Retrievers. Chains combine multiple steps into a pipeline.\n\nRAG combines retrieval with generation for accurate, grounded answers."

def split_text(text, chunk_size=100, chunk_overlap=20):
    """Simple text splitter."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - chunk_overlap
    return chunks

chunks = split_text(sample, chunk_size=100, chunk_overlap=20)
print("Split into " + str(len(chunks)) + " chunks")
for i, chunk in enumerate(chunks):
    print("  Chunk " + str(i) + " (" + str(len(chunk)) + " chars): " + chunk[:60] + "...")

Split into 6 chunks
  Chunk 0 (100 chars): LangChain is a framework for building LLM applications. It p...
  Chunk 1 (100 chars): ompt management, chaining, memory, and more.

LangChain supp...
  Chunk 2 (100 chars): oviders including OpenAI, Anthropic, and Groq. The framework...
  Chunk 3 (100 chars): duction use.

The key abstractions are Chains, Agents, and R...
  Chunk 4 (100 chars): mbine multiple steps into a pipeline.

RAG combines retrieva...
  Chunk 5 (29 chars): r accurate, grounded answers....


## 7. Splitting Documents

In [3]:
doc = Document(page_content=sample, metadata={"source": "example.txt"})

def split_document(doc, chunk_size=100, chunk_overlap=20):
    chunks = split_text(doc.page_content, chunk_size, chunk_overlap)
    return [Document(page_content=c, metadata=doc.metadata) for c in chunks]

docs = split_document(doc, chunk_size=150, chunk_overlap=30)
print("Split into " + str(len(docs)) + " document chunks")
for d in docs:
    print("  -", d.metadata, ":", len(d.page_content), "chars")

Split into 4 document chunks
  - {'source': 'example.txt'} : 150 chars
  - {'source': 'example.txt'} : 150 chars
  - {'source': 'example.txt'} : 150 chars
  - {'source': 'example.txt'} : 69 chars


## 8. Chunk Size Effects

In [4]:
sizes = [50, 100, 200, 500]
for size in sizes:
    chunks = split_text(sample, chunk_size=size, chunk_overlap=10)
    print("Size=" + str(size) + ": " + str(len(chunks)) + " chunks")

Size=50: 11 chunks
Size=100: 5 chunks
Size=200: 3 chunks
Size=500: 1 chunks


## 9. Recursive Splitting

In [5]:
def recursive_split(text, chunk_size=100, chunk_overlap=20, separators=None):
    """Split by separators in order."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " "]
    
    if len(text) <= chunk_size:
        return [text]
    
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current = ""
            for part in parts:
                if len(current) + len(part) + len(sep) <= chunk_size:
                    current += part + sep
                else:
                    if current:
                        chunks.append(current.strip())
                    current = part + sep
            if current:
                chunks.append(current.strip())
            return chunks
    
    return split_text(text, chunk_size, chunk_overlap)

chunks = recursive_split(sample, chunk_size=120)
print("Recursive split: " + str(len(chunks)) + " chunks")
for i, c in enumerate(chunks):
    print("  " + str(i) + ": " + c[:60] + "...")

Recursive split: 4 chunks
  0: LangChain is a framework for building LLM applications. It p...
  1: LangChain supports multiple LLM providers including OpenAI, ...
  2: The key abstractions are Chains, Agents, and Retrievers. Cha...
  3: RAG combines retrieval with generation for accurate, grounde...


## 10. Common Mistakes

1. Chunks too small (lose context)
2. Chunks too large (exceed context)
3. No overlap (miss boundaries)
4. Wrong separator for content type

## 11. Coding Exercises

### Exercise 1: Split a Long Text
Create a 2000-word text and split it.

### Exercise 2: Compare Strategies
Compare different chunk sizes.

In [6]:
# EXERCISE 1
print("Exercise: Split a long document.")

Exercise: Split a long document.


In [7]:
# EXERCISE 2
print("Exercise: Compare chunk size effects.")

Exercise: Compare chunk size effects.


## 12. Closed-Book Recall

1. What does chunk_overlap do?
2. Why use recursive splitting?
3. How do you choose chunk_size?

## 13. Summary

Text splitters break documents into chunks. Recursive splitting tries separators in order. Choose chunk_size based on your use case. Always use overlap.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [langchain-core]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```